# Minor Project: CNN Image Classification and Streamlit Application
**Student Name:** _your name_  |  **Batch:** _your batch_  |  **Date:** _date_

**Dataset:** MNIST handwritten digits (Kaggle: *MNIST in CSV* by oddrationale)  
**Streamlit App Topic:** Health & Fitness Calculator (BMI + daily calories)

---
## Part A - CNN Image Classification
**Objective:** Train a Convolutional Neural Network to recognise handwritten digits 0-9.

**How to run on Kaggle:** New Notebook -> *Add Input* -> search **"MNIST in CSV"** (oddrationale) -> Add. Then Run All.

## Step 0: Import libraries

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

## Step 1: Load the dataset and understand it
The Kaggle dataset has `mnist_train.csv` and `mnist_test.csv`. Each row = 1 label + 784 pixel values (28 x 28 image).

In [ ]:
# Look for the Kaggle CSV files; if not found, fall back to Keras' built-in MNIST (same data).
train_files = glob.glob('/kaggle/input/**/mnist_train.csv', recursive=True)
test_files  = glob.glob('/kaggle/input/**/mnist_test.csv',  recursive=True)

if train_files and test_files:
    print("Loading from Kaggle CSV files...")
    train_df = pd.read_csv(train_files[0])
    test_df  = pd.read_csv(test_files[0])
    y_train = train_df['label'].values
    X_train = train_df.drop('label', axis=1).values.reshape(-1, 28, 28)
    y_test  = test_df['label'].values
    X_test  = test_df.drop('label', axis=1).values.reshape(-1, 28, 28)
else:
    print("Kaggle CSV not found - loading MNIST from Keras instead...")
    (X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("Data loaded successfully!")

In [ ]:
class_names = [str(i) for i in range(10)]

print("Training images shape :", X_train.shape)
print("Training labels shape :", y_train.shape)
print("Test images shape     :", X_test.shape)
print("Test labels shape     :", y_test.shape)
print()
print("Image size            : 28 x 28 pixels, grayscale (1 channel)")
print("Number of classes     :", len(class_names))
print("Class names           :", class_names)
print("Training samples      :", len(X_train))
print("Test samples          :", len(X_test))
print("Pixel value range     :", X_train.min(), "to", X_train.max())

### Sample images from the dataset

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.suptitle("Sample MNIST Training Images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution - is the data balanced?
plt.figure(figsize=(8, 4))
sns.countplot(x=y_train, color='steelblue')
plt.title("Number of training images per digit")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

## Step 2: Preprocess the images and labels
- Scale pixels from 0-255 to 0-1 (helps training)
- Add a channel dimension (28, 28, 1) because CNNs expect it
- Labels stay as integers, so we use `sparse_categorical_crossentropy` (no one-hot needed)

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print("After preprocessing:")
print("X_train shape:", X_train.shape, "| min/max:", X_train.min(), X_train.max())
print("X_test shape :", X_test.shape)

## Step 3: Create and train the CNN model
**Architecture:** 2 convolution + pooling blocks to find patterns (edges, curves), then a Dense layer to decide the digit. Dropout reduces overfitting.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()   # <-- take a screenshot of this output for the report

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,   # 10% of training data used for validation
    verbose=1
)

### Training and validation accuracy / loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['accuracy'], marker='o', label='Training accuracy')
plt.plot(history.history['val_accuracy'], marker='o', label='Validation accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], marker='o', label='Training loss')
plt.plot(history.history['val_loss'], marker='o', label='Validation loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 4: Evaluate the model using test data

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc * 100:.2f}%")

## Step 5: Show sample actual and predicted labels

In [ ]:
pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(pred_probs, axis=1)

plt.figure(figsize=(12, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    plt.title(f"Actual: {y_test[i]}\nPredicted: {y_pred[i]}", color=color)
    plt.axis('off')
plt.suptitle("Actual vs Predicted (green = correct, red = wrong)", fontsize=14)
plt.tight_layout()
plt.show()

### Correct and incorrect predictions

In [ ]:
correct_idx   = np.where(y_pred == y_test)[0]
incorrect_idx = np.where(y_pred != y_test)[0]
print("Correct predictions  :", len(correct_idx))
print("Incorrect predictions:", len(incorrect_idx))

# Show 10 correct predictions
plt.figure(figsize=(12, 3))
for n, i in enumerate(correct_idx[:10]):
    plt.subplot(2, 5, n + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    plt.title(f"Actual {y_test[i]} | Pred {y_pred[i]}", color='green', fontsize=9)
    plt.axis('off')
plt.suptitle("Correct Predictions"); plt.tight_layout(); plt.show()

# Show 10 incorrect predictions
plt.figure(figsize=(12, 3))
for n, i in enumerate(incorrect_idx[:10]):
    plt.subplot(2, 5, n + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    plt.title(f"Actual {y_test[i]} | Pred {y_pred[i]}", color='red', fontsize=9)
    plt.axis('off')
plt.suptitle("Incorrect Predictions"); plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix + per-class report
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

## Step 6: Explain the result in simple words

In [ ]:
final_train_acc = history.history['accuracy'][-1] * 100
final_val_acc   = history.history['val_accuracy'][-1] * 100
print(f"""SUMMARY
-------
- The CNN learned to recognise handwritten digits from {len(X_train)} training images.
- Final training accuracy  : {final_train_acc:.2f}%
- Final validation accuracy: {final_val_acc:.2f}%
- Test accuracy (unseen)   : {test_acc*100:.2f}%
- Out of {len(X_test)} test images, {len(incorrect_idx)} were predicted wrongly.
- Training and validation accuracy are close, so the model is NOT badly overfitting.
- Mistakes mostly happen on messy or unclear handwriting (e.g. 4 vs 9, 3 vs 5, 7 vs 2).
""")

### Challenges and possible improvements
**Challenges**
- Some handwritten digits are very unclear, even for humans.
- Choosing the number of epochs: too few = underfit, too many = overfit.
- Making sure image shape `(28, 28, 1)` matches what the CNN expects.

**Possible improvements**
- Add **data augmentation** (small rotations, shifts, zoom).
- Add **BatchNormalization** and more convolution layers.
- Use **EarlyStopping** and learning-rate scheduling.
- Try **CIFAR-10** for a harder colour-image task.
- Tune hyperparameters (filters, dropout, batch size).

---
## Part B - Streamlit Application: Health & Fitness Calculator
**Purpose:** Calculates BMI, BMI category, and estimated daily calories (BMR x activity).  
**Inputs:** name, age, gender, height, weight, activity level.  
**Output:** BMI value, category, daily calorie need, and a simple tip.  
**Invalid-input handling:** rejects zero/unrealistic height, weight and age.

The cell below saves the app to a file. In your **final submission**, paste Part A (CNN) and Part B (app) into **one** `.py` file as the guide requires.

In [ ]:
%%writefile health_app.py
import streamlit as st

# ---------------- PART B: STREAMLIT APP ----------------
st.set_page_config(page_title="Health & Fitness Calculator", page_icon="💪")

st.title("💪 Health & Fitness Calculator")
st.write("Enter your details in the sidebar to calculate your **BMI** and **daily calorie needs**.")


def calculate_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    return weight_kg / (height_m ** 2)


def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight", "Consider a nutrient-rich diet with more calories."
    elif bmi < 25:
        return "Normal weight", "Great! Keep up your balanced diet and exercise."
    elif bmi < 30:
        return "Overweight", "Try regular exercise and watch your portion sizes."
    return "Obese", "Consider talking to a doctor or dietitian for a plan."


def calculate_bmr(weight, height, age, gender):
    # Mifflin-St Jeor equation
    base = 10 * weight + 6.25 * height - 5 * age
    return base + 5 if gender == "Male" else base - 161


activity_levels = {
    "Sedentary (little or no exercise)": 1.2,
    "Light (1-3 days/week)": 1.375,
    "Moderate (3-5 days/week)": 1.55,
    "Active (6-7 days/week)": 1.725,
}

# ---------------- User inputs ----------------
st.sidebar.header("Your Details")
name = st.sidebar.text_input("Name")
age = st.sidebar.number_input("Age (years)", min_value=0, max_value=120, value=20)
gender = st.sidebar.radio("Gender", ["Male", "Female"])
height = st.sidebar.number_input("Height (cm)", min_value=0.0, max_value=250.0, value=170.0)
weight = st.sidebar.number_input("Weight (kg)", min_value=0.0, max_value=300.0, value=65.0)
activity = st.sidebar.selectbox("Activity level", list(activity_levels.keys()))

# ---------------- Processing + output ----------------
if st.sidebar.button("Calculate"):
    # Basic invalid-input handling
    if age < 10 or age > 100:
        st.error("Please enter an age between 10 and 100.")
    elif height < 100 or height > 250:
        st.error("Please enter a realistic height between 100 and 250 cm.")
    elif weight < 20 or weight > 300:
        st.error("Please enter a realistic weight between 20 and 300 kg.")
    else:
        bmi = calculate_bmi(weight, height)
        category, tip = bmi_category(bmi)
        calories = calculate_bmr(weight, height, age, gender) * activity_levels[activity]

        st.subheader(f"Results for {name if name.strip() else 'you'}")
        col1, col2 = st.columns(2)
        col1.metric("BMI", f"{bmi:.1f}")
        col2.metric("Daily Calories", f"{calories:.0f} kcal")
        st.info(f"**Category:** {category}")
        st.success(f"💡 Tip: {tip}")
else:
    st.write("👈 Fill in your details and click **Calculate**.")

To run the app on your computer (not on Kaggle):
```
pip install streamlit
streamlit run health_app.py
```